In [ ]:
#make moon example
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(device)

data, _ = make_moons(n_samples=5000, noise=0.05, random_state=42)
data_tensor = torch.tensor(data).float()
print(data_tensor.shape)
print(data_tensor.dtype)
#only x, only y
plt.scatter(data_tensor[:, 0],data_tensor[:, 1], s=2, label = "data")

#5000 points, 2 dimension
noise = torch.randn(5000, 2)
plt.scatter(noise[:, 0], noise[:, 1], s=2, label="noise")

plt.legend()
plt.show()


In [ ]:
import torch
import torch.nn as nn
class VelocityMLP(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(3, 128)
        self.relu = nn.ReLU()
        self.layer2 = nn.Linear(128, 128)
        self.layer3 = nn.Linear(128, 2)

    def forward(self, x_t, t):
        x = torch.cat([x_t, t], dim=1) 
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        x = self.relu(x)
        x = self.layer3(x)
        return x

model = VelocityMLP()
x_t = torch.randn(64, 2)
t = torch.rand(64, 1)
print(model(x_t, t).shape)


In [ ]:
import torch

#examplar training - flow matching
model = VelocityMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
data = data_tensor.to(device)

for step in range(5000):

    #torch.randint(min, max, shape)
    idx = torch.randint(0, 5000, (256,))

    #parameters
    x1 = data[idx]
    x0 = torch.randn_like(x1)
    t = torch.rand(256, 1, device=device)

    x_t = (1-t)*x0 + t*x1
    target = x1 - x0
    pred = model(x_t, t)
    loss = ((pred-target)**2).mean()

    #3stepsss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 500 == 0:
        print(f"step {step}  |  loss {loss.item():.4f}")
